# HomeReach Melbourne — 01: Rental workbook audit

**Week 2, milestone 1: understand the official source before cleaning it.**

This notebook is tailored to:
`quarterly-median-rents-local-government-area-september-quarter-2025-excel.xlsx`

**Selected worksheet:** `2br Flat`  
**Exact category label in cell A2:** `2 bedroom flats`

Read each explanation and run the following code cell, working from top to bottom.
Use the **homereach** Python environment. All code runs locally; this notebook does
not need an API key, internet access, GitHub access, or Streamlit.

The source workbook is read-only throughout. The final cell writes an audit report
under `docs/` and small audit tables under `data/interim/rent_audit/`. It does **not**
write a cleaned rental panel, change the dashboard, or fit a model.

**Source locations:** worksheet names are from the workbook tab list; layout is
`2br Flat!A1:HF3`; area metadata is `A4:B95`; metropolitan blocks are `A57:HF90`.
The publisher's page is https://www.dffh.vic.gov.au/publications/rental-report.

## 1. Confirm the Python environment

A notebook's **kernel** is the Python process that runs its cells. Selecting a
terminal environment does not always select the same notebook kernel. On your
Windows computer, the Python path below should include `envs\homereach`.

`Path` handles file paths; `hashlib` creates a file fingerprint; pandas reads and
inspects the table. We are not installing anything here.

In [2]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import numbers
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 50)

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Current folder:", Path.cwd())

Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3
Current folder: d:\My project\homereach_melbourne\notebooks


## 2. Find the unchanged workbook

The root-finding function searches the current directory and its parents for the
project's `pyproject.toml` and `data` directory. This works when the notebook starts
from either the repository root or its `notebooks` folder.

Keep the original Excel filename. The SHA-256 fingerprint identifies the exact
file contents; we compare it again at the end to verify the source did not change.

In [3]:
def find_project_root(start: Path) -> Path:
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "data").is_dir():
            return folder
    raise FileNotFoundError(
        "Open this notebook from the HomeReach project folder or its notebooks folder. "
        "The project root must contain pyproject.toml and data/."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
WORKBOOK_NAME = (
    "quarterly-median-rents-local-government-area-"
    "september-quarter-2025-excel.xlsx"
)
WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "rent" / WORKBOOK_NAME
SHEET_NAME = "2br Flat"

print("Project root:", PROJECT_ROOT)
print("Workbook found:", WORKBOOK_PATH.is_file())
print("Workbook filename:", WORKBOOK_PATH.name)

if not WORKBOOK_PATH.is_file():
    available = sorted(p.name for p in WORKBOOK_PATH.parent.glob("*.xlsx"))
    raise FileNotFoundError(
        f"Cannot find {WORKBOOK_PATH}. Excel files in this folder: {available}"
    )

source_sha256 = hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest()
print("File size (bytes):", WORKBOOK_PATH.stat().st_size)
print("SHA-256:", source_sha256)

Project root: D:\My project\homereach_melbourne
Workbook found: True
Workbook filename: quarterly-median-rents-local-government-area-september-quarter-2025-excel.xlsx
File size (bytes): 674773
SHA-256: b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837


## 3. List sheets and load the selected sheet without guessing headers

`header=None` retains all worksheet rows, including the title and both header rows.
`dtype=object` and `keep_default_na=False` help preserve the source values for
inspection: a hyphen stays `-` and a blank stays an empty string.

This source has **seven** worksheets. Be exact with `2br Flat`, including its case.
The first sheet is `1br flat`, so relying on the default first-sheet read would
analyse the wrong property category.

Reference: https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html

In [4]:
with pd.ExcelFile(WORKBOOK_PATH, engine="openpyxl") as excel:
    sheet_names = excel.sheet_names
    if SHEET_NAME not in sheet_names:
        raise ValueError(f"Expected {SHEET_NAME!r}; found {sheet_names}")
    raw = pd.read_excel(
        excel,
        sheet_name=SHEET_NAME,
        header=None,
        dtype=object,
        keep_default_na=False,
    )

print("Worksheets:")
for number, name in enumerate(sheet_names, start=1):
    print(f"{number}. {name}")
print("Selected sheet:", SHEET_NAME)
print("Raw sheet shape (rows, columns):", raw.shape)
print("Property label from Excel A2:", raw.iat[1, 0])

Worksheets:
1. 1br flat
2. 2br Flat
3. 3br Flat
4. 2br House
5. 3br House
6. 4br House
7. All Properties
Selected sheet: 2br Flat
Raw sheet shape (rows, columns): (95, 214)
Property label from Excel A2: 2 bedroom flats


## 4. Inspect the layout using Excel-style coordinates

Excel row numbers begin at **1**; pandas `.iloc` positions begin at **0**.
For example, Excel row 2 is `raw.iloc[1]`.

In this workbook: row 1 is the title; row 2 contains repeated quarter labels;
row 3 alternates `Count` and `Median`; records start at row 4.
Column A identifies a regional group and column B identifies an LGA or total row.
Do not treat `Count` as a dollar value or the title as a normal column header.

In [5]:
def excel_column_name(position: int) -> str:
    """Convert a zero-based column position to an Excel column label."""
    if position < 0:
        raise ValueError("Column position cannot be negative.")
    number = position + 1
    letters = ""
    while number:
        number, remainder = divmod(number - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters


def excel_preview(frame: pd.DataFrame, row_positions, column_positions) -> pd.DataFrame:
    row_positions = list(row_positions)
    column_positions = list(column_positions)
    view = frame.iloc[row_positions, column_positions].copy()
    view.index = [r + 1 for r in row_positions]
    view.index.name = "Excel row"
    view.columns = [excel_column_name(c) for c in column_positions]
    return view

print("Title, header rows and first records — A1:J10")
display(excel_preview(raw, range(10), range(10)))

print("Latest two quarters — rows 1 to 7, columns HC:HF")
display(excel_preview(raw, range(7), range(raw.shape[1] - 4, raw.shape[1])))

print("Metropolitan sections begin at row 57 — A57:F65")
display(excel_preview(raw, range(56, 65), range(6)))

print("The bottom of the sheet includes aggregate rows — A87:F95")
display(excel_preview(raw, range(86, 95), range(6)))

Title, header rows and first records — A1:J10


,A,B,C,D,E,F,G,H,I,J
Excel row,,,,,,,,,,
1,Quarterly median rents by Local Government Area,,,,,,,,,
2,2 bedroom flats,,Jun 1999,Jun 1999,Sep 1999,Sep 1999,Dec 1999,Dec 1999,Mar 2000,Mar 2000
3,,,Count,Median,Count,Median,Count,Median,Count,Median
4,Barwon South West,Colac-Otway,18,110,14,110,12,110,20,110
5,,Corangamite,-,-,-,-,-,-,6,113
6,,Glenelg,20,100,27,90,25,100,38,95
7,,Greater Geelong,386,125,352,130,390,130,419,130
8,,Moyne,-,-,-,-,-,-,5,110
9,,Queenscliffe,-,-,-,-,6,145,6,138


Latest two quarters — rows 1 to 7, columns HC:HF


,HC,HD,HE,HF
Excel row,,,,
1,,,,
2,Jun 2025,Jun 2025,Sep 2025,Sep 2025
3,Count,Median,Count,Median
4,11,390,13,370
5,9,350,-,-
6,19,360,22,305
7,261,430,257,435


Metropolitan sections begin at row 57 — A57:F65


,A,B,C,D,E,F
Excel row,,,,,,
57,North and West Metro,Banyule,214,165,202,165
58,,Brimbank,159,120,157,130
59,,Darebin,379,145,380,145
60,,Hobsons Bay,212,150,242,155
61,,Hume,97,150,93,150
62,,Maribyrnong,178,120,165,125
63,,Melbourne,486,290,532,280
64,,Melton,24,135,28,128
65,,Merri-bek,394,140,399,150


The bottom of the sheet includes aggregate rows — A87:F95


,A,B,C,D,E,F
Excel row,,,,,,
87,,Mornington Penin'a,199,120,165,125
88,,Port Phillip,796,220,836,225
89,,Stonnington,540,210,562,210
90,,Group Total,3372,165,3530,170
91,Table Total,Victoria,10059,150,10269,152
92,,,,,,
93,METRO NON-METRO,Metro,7904,160,8165,165
94,,Non-Metro,2155,120,2104,120
95,,Victoria,10059,150,10269,152


## 5. Audit every quarter / measure pair

From column C onward, each quarter occupies two adjacent columns: **Count, then
Median**. C2/D2 contain `Jun 1999`; HE2/HF2 contain `Sep 2025`.

The audit validates that each pair has matching dates and the expected measure
labels. It then checks for duplicate or missing quarters. The column map will be
useful when we write the cleaning pipeline; it prevents count and rent values from
being mixed together.

The actual first quarter here is **June 1999**, not June 2000. We use the file's
observed coverage rather than a previously assumed start date.

In [6]:
quarter_rows = []
measurement_positions = list(range(2, raw.shape[1]))
if len(measurement_positions) % 2:
    raise ValueError("An unpaired measurement column was found; inspect the headers.")

# Explicit quarter-end months avoid depending on the operating system's language.
quarter_months = {"Mar": 3, "Jun": 6, "Sep": 9, "Dec": 12}

for count_pos in range(2, raw.shape[1], 2):
    median_pos = count_pos + 1
    count_label = str(raw.iat[1, count_pos]).strip()
    median_label = str(raw.iat[1, median_pos]).strip()
    measures = (
        str(raw.iat[2, count_pos]).strip(),
        str(raw.iat[2, median_pos]).strip(),
    )
    if count_label != median_label or measures != ("Count", "Median"):
        raise ValueError(f"Unexpected column pair at {excel_column_name(count_pos)}.")

    month_text, year_text = count_label.split()
    if month_text not in quarter_months:
        raise ValueError(f"Unrecognised quarter label: {count_label}")
    period = pd.Period(
        year=int(year_text), month=quarter_months[month_text], freq="Q"
    )
    quarter_rows.append({
        "source_quarter_label": count_label,
        "quarter": str(period),
        "count_column_position": count_pos,
        "median_column_position": median_pos,
        "count_excel_column": excel_column_name(count_pos),
        "median_excel_column": excel_column_name(median_pos),
    })

quarter_map = pd.DataFrame(quarter_rows)
periods = pd.PeriodIndex(quarter_map["quarter"], freq="Q")
continuous_periods = pd.period_range(periods.min(), periods.max(), freq="Q")

if periods.has_duplicates or not periods.equals(continuous_periods):
    raise ValueError("The quarter sequence has duplicates, gaps or ordering problems.")

print("First quarter:", quarter_map.iloc[0]["source_quarter_label"])
print("Last quarter:", quarter_map.iloc[-1]["source_quarter_label"])
print("Quarter count:", len(quarter_map))
print("Continuous quarter sequence: True")
display(pd.concat([quarter_map.head(3), quarter_map.tail(3)], ignore_index=True))

First quarter: Jun 1999
Last quarter: Sep 2025
Quarter count: 106
Continuous quarter sequence: True


,source_quarter_label,quarter,count_column_position,median_column_position,count_excel_column,median_excel_column
0,Jun 1999,1999Q2,2,3,C,D
1,Sep 1999,1999Q3,4,5,E,F
2,Dec 1999,1999Q4,6,7,G,H
3,Mar 2025,2025Q1,208,209,HA,HB
4,Jun 2025,2025Q2,210,211,HC,HD
5,Sep 2025,2025Q3,212,213,HE,HF


## 6. Separate LGAs, regional labels and totals

Column A gives the regional label only on the first row of each group. Filling
that label down is appropriate **only for this metadata column**. We do not
forward-fill missing counts or rent values.

The bottom of the sheet contains `Metro`, `Non-Metro` and `Victoria` totals, and
each regional group has `Group Total`. These must not be counted as individual
LGAs. The `Victoria` total also appears twice in this sheet.

We preserve source names exactly, including `Mornington Penin'a`. Any later
matching to official geographic codes needs a reviewed crosswalk, not guessed
renaming. Also distinguish the regional group `Hume` from the metropolitan LGA
named `Hume`.

In [7]:
metadata = raw.iloc[3:, :2].copy()
metadata.columns = ["source_region", "lga_name"]
metadata["excel_row"] = metadata.index + 1

for column in ["source_region", "lga_name"]:
    metadata[column] = metadata[column].astype("string").str.strip().replace("", pd.NA)

metadata["source_region"] = metadata["source_region"].ffill()
aggregate_labels = {"Group Total", "Metro", "Non-Metro", "Victoria"}

aggregate_rows = metadata.loc[metadata["lga_name"].isin(aggregate_labels)].copy()
all_lgas = metadata.loc[
    metadata["lga_name"].notna() & ~metadata["lga_name"].isin(aggregate_labels)
].copy()

if all_lgas["lga_name"].duplicated().any():
    raise ValueError("Duplicate LGA source labels require manual review.")

METRO_REGIONS = ["North and West Metro", "Eastern Metro", "Southern Metro"]
metro_lgas = all_lgas.loc[all_lgas["source_region"].isin(METRO_REGIONS)].copy()

print("Individual LGAs in the complete sheet:", len(all_lgas))
print("Aggregate rows excluded:", len(aggregate_rows))
print("LGAs in the selected metropolitan groups:", len(metro_lgas))

group_summary = (
    all_lgas.groupby("source_region", sort=False)
    .size().reset_index(name="lga_count")
)
display(group_summary)
print("Metropolitan LGA inventory (source spelling preserved)")
display(metro_lgas.reset_index(drop=True))

Individual LGAs in the complete sheet: 79
Aggregate rows excluded: 12
LGAs in the selected metropolitan groups: 31


,source_region,lga_count
0,Barwon South West,9
1,Grampians,11
2,Loddon Mallee,10
3,Hume,12
4,Gippsland,6
5,North and West Metro,14
6,Eastern Metro,7
7,Southern Metro,10


Metropolitan LGA inventory (source spelling preserved)


,source_region,lga_name,excel_row
0,North and West Metro,Banyule,57
1,North and West Metro,Brimbank,58
2,North and West Metro,Darebin,59
3,North and West Metro,Hobsons Bay,60
4,North and West Metro,Hume,61
5,North and West Metro,Maribyrnong,62
6,North and West Metro,Melbourne,63
7,North and West Metro,Melton,64
8,North and West Metro,Merri-bek,65
9,North and West Metro,Moonee Valley,66


## 7. Audit numeric values and missing markers

A hyphen (`-`) is a source **non-numeric/unavailable marker**, not a rent of $0.
We count the marker without guessing its precise suppression reason or threshold.

Check both the full Victorian LGA set and the selected metropolitan subset, and
check `Count` separately from `Median`. Absence of missing values in one subset is
not proof that every aspect of the data is correct.

In [8]:
def audit_values(row_indices, column_positions, scope: str, measure: str) -> dict:
    values = raw.loc[row_indices].iloc[:, column_positions].to_numpy().ravel().tolist()
    numeric_values = []
    blank_count = 0
    text_markers = Counter()

    for value in values:
        if pd.isna(value) or (isinstance(value, str) and not value.strip()):
            blank_count += 1
        elif isinstance(value, numbers.Real) and not isinstance(value, bool):
            numeric_values.append(float(value))
        else:
            text_markers[str(value).strip()] += 1

    unexpected = {text: count for text, count in text_markers.items() if text != "-"}
    return {
        "scope": scope,
        "measure": measure,
        "expected_cells": len(values),
        "numeric_cells": len(numeric_values),
        "hyphen_cells": text_markers.get("-", 0),
        "blank_cells": blank_count,
        "unexpected_text": json.dumps(unexpected, ensure_ascii=False),
        "zero_cells": sum(value == 0 for value in numeric_values),
        "negative_cells": sum(value < 0 for value in numeric_values),
    }

quality_rows = []
for scope, area_table in [("All 79 source LGAs", all_lgas), ("Metro 31 LGAs", metro_lgas)]:
    for measure, map_column in [
        ("Count", "count_column_position"),
        ("Median", "median_column_position"),
    ]:
        quality_rows.append(audit_values(
            area_table.index, quarter_map[map_column].tolist(), scope, measure
        ))

quality_audit = pd.DataFrame(quality_rows)
display(quality_audit)

metro_quality = quality_audit.loc[quality_audit["scope"].eq("Metro 31 LGAs")]
expected_panel_rows = len(metro_lgas) * len(quarter_map)
print("Potential metro LGA-quarter rows:", expected_panel_rows)
print("This is a future tidy-panel size, not the size of the raw Excel sheet.")

,scope,measure,expected_cells,numeric_cells,hyphen_cells,blank_cells,unexpected_text,zero_cells,negative_cells
0,All 79 source LGAs,Count,8374,7237,1137,0,{},0,0
1,All 79 source LGAs,Median,8374,7237,1137,0,{},0,0
2,Metro 31 LGAs,Count,3286,3286,0,0,{},0,0
3,Metro 31 LGAs,Median,3286,3286,0,0,{},0,0


Potential metro LGA-quarter rows: 3286
This is a future tidy-panel size, not the size of the raw Excel sheet.


## 8. Trace three latest-quarter examples back to Excel

These are a **data-reading check**, not recommendations or currently available
rental listings. Keep the reporting quarter and property category with every
value. `Melbourne` here names an LGA, not all metropolitan Melbourne.

Source cells for this file: `2br Flat!HE63:HF63` (Melbourne),
`HE76:HF76` (Monash), and `HE77:HF77` (Whitehorse).

In [9]:
last_pair = quarter_map.iloc[-1]
examples = metro_lgas.loc[
    metro_lgas["lga_name"].isin(["Melbourne", "Monash", "Whitehorse"])
].copy()
examples["reporting_quarter"] = last_pair["source_quarter_label"]
examples["published_count"] = raw.loc[examples.index].iloc[
    :, int(last_pair["count_column_position"])
].to_numpy()
examples["median_weekly_rent_aud"] = raw.loc[examples.index].iloc[
    :, int(last_pair["median_column_position"])
].to_numpy()
examples["source_cells"] = examples["excel_row"].map(
    lambda row: f"{last_pair['count_excel_column']}{row}:"
                f"{last_pair['median_excel_column']}{row}"
)
display(examples[[
    "lga_name", "reporting_quarter", "published_count",
    "median_weekly_rent_aud", "source_cells"
]].reset_index(drop=True))

,lga_name,reporting_quarter,published_count,median_weekly_rent_aud,source_cells
0,Melbourne,Sep 2025,3209,730,HE63:HF63
1,Monash,Sep 2025,421,600,HE76:HF76
2,Whitehorse,Sep 2025,464,600,HE77:HF77


## 9. Verify this exact workbook snapshot

These assertions document the observed September 2025 workbook. A later workbook
may legitimately have more quarters or a different layout, so a failed assertion
means **re-audit the source**, not change the data to make a test pass.

The missing-marker figures below exclude all aggregate rows.

In [10]:
assert len(sheet_names) == 7, "The workbook sheet inventory changed."
assert raw.shape == (95, 214), "The selected sheet's layout changed."
assert str(raw.iat[1, 0]).strip() == "2 bedroom flats"
assert len(quarter_map) == 106
assert quarter_map.iloc[0]["quarter"] == "1999Q2"
assert quarter_map.iloc[-1]["quarter"] == "2025Q3"
assert len(all_lgas) == 79
assert len(aggregate_rows) == 12
assert len(metro_lgas) == 31
assert expected_panel_rows == 3286
assert all(quality_audit["unexpected_text"].eq("{}"))
assert all(quality_audit["negative_cells"].eq(0))
assert all(metro_quality["numeric_cells"].eq(3286))
assert all(metro_quality["hyphen_cells"].eq(0))
assert all(metro_quality["blank_cells"].eq(0))
assert all(quality_audit.loc[
    quality_audit["scope"].eq("All 79 source LGAs"), "hyphen_cells"
].eq(1137))
assert hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest() == source_sha256

print("AUDIT CHECKPOINT PASSED")
print("Selected sheet: 2br Flat")
print("Quarters: 106 (Jun 1999 to Sep 2025)")
print("Metro LGAs: 31")
print("Expected tidy rows: 3286")
print("Metro missing count values: 0")
print("Metro missing median values: 0")
print("Original workbook unchanged: True")

AUDIT CHECKPOINT PASSED
Selected sheet: 2br Flat
Quarters: 106 (Jun 1999 to Sep 2025)
Metro LGAs: 31
Expected tidy rows: 3286
Metro missing count values: 0
Metro missing median values: 0
Original workbook unchanged: True


## 10. Save audit evidence only

The next cell writes these derived files:

- `docs/rental_workbook_audit.md`: a human-readable report for the repository.
- `data/interim/rent_audit/quarter_header_map.csv`: where every quarter's two measures live.
- `data/interim/rent_audit/metro_lga_inventory.csv`: source labels and row positions.
- `data/interim/rent_audit/value_quality_audit.csv`: numeric / marker counts.
- `data/interim/rent_audit/audit_summary.json`: machine-readable audit metadata.

Re-running replaces **only these generated audit outputs**, not the original
workbook or your Week 1 documents. Local absolute paths and your Python executable
are not included in the saved report.

In [11]:
audit_dir = PROJECT_ROOT / "data" / "interim" / "rent_audit"
audit_dir.mkdir(parents=True, exist_ok=True)
report_path = PROJECT_ROOT / "docs" / "rental_workbook_audit.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

summary = {
    "source_filename": WORKBOOK_NAME,
    "source_relative_path": WORKBOOK_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "source_page": "https://www.dffh.vic.gov.au/publications/rental-report",
    "source_sha256": source_sha256,
    "audit_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "worksheet_names": sheet_names,
    "selected_sheet": SHEET_NAME,
    "source_property_label": str(raw.iat[1, 0]),
    "raw_shape": list(raw.shape),
    "title_excel_row": 1,
    "quarter_header_excel_row": 2,
    "measure_header_excel_row": 3,
    "data_start_excel_row": 4,
    "first_quarter": str(quarter_map.iloc[0]["quarter"]),
    "last_quarter": str(quarter_map.iloc[-1]["quarter"]),
    "quarter_count": len(quarter_map),
    "all_lga_count": len(all_lgas),
    "aggregate_rows_excluded": len(aggregate_rows),
    "metro_region_labels": METRO_REGIONS,
    "metro_lga_count": len(metro_lgas),
    "expected_metro_tidy_rows": expected_panel_rows,
    "quality_audit": quality_audit.to_dict(orient="records"),
    "status": "audit_passed_not_yet_cleaned",
}

quarter_map.to_csv(audit_dir / "quarter_header_map.csv", index=False)
metro_lgas.reset_index(drop=True).to_csv(audit_dir / "metro_lga_inventory.csv", index=False)
quality_audit.to_csv(audit_dir / "value_quality_audit.csv", index=False)
(audit_dir / "audit_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

report_lines = [
    "# Rental workbook audit — HomeReach Melbourne", "",
    "## Source and purpose", "",
    f"- Source file: `{WORKBOOK_NAME}`",
    f"- Source SHA-256: `{source_sha256}`",
    f"- Audited UTC: {summary['audit_timestamp_utc']}",
    "- Publisher page: https://www.dffh.vic.gov.au/publications/rental-report",
    "- Purpose: source inspection before implementing the cleaning pipeline.",
    "- No Excel source cells were edited.", "",
    "## Workbook structure", "",
    "Worksheets: " + ", ".join(f"`{name}`" for name in sheet_names) + ".", "",
    f"Selected worksheet: `{SHEET_NAME}`; category label: `2 bedroom flats` (A2).",
    f"Raw selected-sheet shape: {raw.shape[0]} rows by {raw.shape[1]} columns.",
    "Row 1 is a title, row 2 contains quarters, row 3 contains Count/Median labels,",
    "and records start at row 4. Column A is a regional label; column B is an LGA or total.",
    "The first measure pair is C:D and the final pair is HE:HF.", "",
    "## Time and geographic coverage", "",
    "- 106 continuous quarters, June 1999 (1999Q2) through September 2025 (2025Q3).",
    "- 79 individual LGAs in the full worksheet (B4:B95 after excluding aggregates).",
    "- 12 aggregate rows excluded: regional Group Totals, Metro, Non-Metro and Victoria.",
    "- 31 LGAs in the source's three metropolitan groups (rows 57:90, totals excluded).",
    "- Group sizes: North and West Metro 14; Eastern Metro 7; Southern Metro 10.",
    "- Planned tidy panel: 31 LGAs x 106 quarters x 1 category = 3,286 rows.", "",
    "## Value audit", "",
    "Across the 79 individual LGAs, each measure has 8,374 cells: 7,237 numeric and",
    "1,137 hyphen markers. The metropolitan subset has 3,286 numeric Count values",
    "and 3,286 numeric Median values, with no blanks or hyphen markers in either.",
    "No negative values or unexpected text markers were found in these measure cells.",
    "These checks are not proof of complete data quality or suitability for forecasting.", "",
    "## Decisions for the cleaning pipeline", "",
    "1. Preserve the source file, published category and original LGA labels.",
    "2. Forward-fill regional metadata only; never forward-fill rents or counts.",
    "3. Remove aggregate rows from the LGA panel; retain them separately if needed.",
    "4. Keep Count and Median separate; do not treat counts as currency.",
    "5. Treat '-' as unavailable, not zero; do not infer a suppression threshold here.",
    "6. Use the three exact source metropolitan group labels for the initial subset.",
    "7. Review names such as Mornington Penin'a before later geographic matching.",
    "8. Do not obtain a metropolitan median by averaging the individual LGA medians.",
    "9. A future forecast must respect what was available at each forecast date;",
    "   a current transport snapshot must not be inserted into historical model evaluation.", "",
    "## Interpretation limits", "",
    "This is a September 2025 reporting snapshot, not live rental listings or current prices.",
    "The source's metropolitan groups are the analysis boundary for this stage; later ABS",
    "boundary matching must document classification and geographic vintage differences.",
    "The Count field is retained as a published observation-count measure; consult the",
    "publisher's definitions before claiming it measures available stock or vacancy.",
    "Audit completed; no final cleaned panel or predictive model has been created.", "",
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")
assert hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest() == source_sha256
print("Saved:", report_path.relative_to(PROJECT_ROOT))
print("Saved audit tables under:", audit_dir.relative_to(PROJECT_ROOT))
print("Source workbook unchanged: True")

Saved: docs\rental_workbook_audit.md
Saved audit tables under: data\interim\rent_audit
Source workbook unchanged: True


## Checkpoint before the cleaning notebook

You should be able to explain why `header=0` would be wrong, why each quarter has
both Count and Median columns, why totals are excluded, and why a blank regional
label can be filled down but a missing rent cannot.

Save this notebook. The next step is to convert the audited metropolitan data
into one row per **LGA × quarter × property category**, with Count and Median in
separate fields. Do not run the earlier generic `melt_quarter_matrix` helper on
this unprocessed two-header-row sheet: it does not yet handle this layout.

References:
- pandas Excel loading: https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html
- VS Code notebook kernels/cells: https://code.visualstudio.com/docs/datascience/jupyter-notebooks
- Publisher definitions: https://www.dffh.vic.gov.au/publications/rental-report